# Modeling and Evaluation

In this section, we will build and evaluate our machine learning models. We will use the processed data from the previous steps and apply various modeling techniques to find the best-performing model for our task.


<a id="top"></a>
## Table of Contents

1. [Setup & Imports](#sec-1-imports)
2. [Load Dataset](#sec-2-load)
3. [Metrics & Utilities](#sec-3-utils)
4. [Model Baselines](#sec-4-baselines)
   - [4.1 Linear Regression](#sec-4-1-lin)
   - [4.2 Random Forest](#sec-4-2-rf)
5. [Model Comparison](#sec-5-compare)
6. [Submission File](#sec-6-submission)
7. [Kaggle Submission](#sec-7-kaggle)

[Back to top](#top)


<a id="sec-1-imports"></a>
## 1. Import Libraries


In [1]:
# --- Imports ---
import os
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE
from sklearn.base import clone


import os
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.metrics import mean_absolute_error, r2_score

<a id="sec-2-load"></a>
## 2. Load Dataset


In [2]:

# --- Define paths ---
data_dir = "../data/"
encoded_dir = os.path.join(data_dir, "feature_selection")

# --- Load encoded & scaled datasets ---
x_train = pd.read_csv(os.path.join(encoded_dir, "20_x_train.csv"))
y_train = pd.read_csv(os.path.join(encoded_dir, "20_y_train.csv")).squeeze()

x_val = pd.read_csv(os.path.join(encoded_dir, "20_x_val.csv"))
y_val = pd.read_csv(os.path.join(encoded_dir, "20_y_val.csv")).squeeze()

x_test = pd.read_csv(os.path.join(encoded_dir, "20_x_test.csv"))

# --- Sanity checks ---
print("Encoded datasets successfully loaded!")
print(f"x_train shape: {x_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"x_val shape:   {x_val.shape}")
print(f"y_val shape:   {y_val.shape}")
print(f"x_test shape:  {x_test.shape}")



Encoded datasets successfully loaded!
x_train shape: (59455, 22)
y_train shape: (59455,)
x_val shape:   (15012, 22)
y_val shape:   (15012,)
x_test shape:  (32567, 22)


<a id="sec-2-load"></a>
## 3. Metrics & Utilities


We use MAE because Kaggle for this course usually evaluates with MAE / it is robust to outliers in price.

In [3]:
# ======================================================
# Utilities: metrics & evaluation
# ======================================================

def evaluate_regression(y_true, y_pred):
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))
    return {"MAE": mae, "R2": r2}

def report_model(name, y_true, y_pred):
    m = evaluate_regression(y_true, y_pred)
    print(f"[{name}]  MAE: {m['MAE']:,.2f} | R²: {m['R2']:.4f}")
    return m


<a id="sec-2-load"></a>
## 4. Model Baselines


In [4]:
# ======================================================
# Baseline Model — Linear Regression
# ======================================================
from sklearn.linear_model import LinearRegression

lin = LinearRegression()
lin.fit(x_train, y_train)

y_val_pred_lin = lin.predict(x_val)
metrics_lin = report_model("LinearRegression (baseline)", y_val, y_val_pred_lin)


[LinearRegression (baseline)]  MAE: 2,908.69 | R²: 0.7863


In [5]:
# ======================================================
# Baseline Model - Ridge Regression
# ======================================================
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)
ridge.fit(x_train, y_train)
y_val_pred_ridge = ridge.predict(x_val)
metrics_ridge = report_model("Ridge (baseline)", y_val, y_val_pred_ridge)


[Ridge (baseline)]  MAE: 2,908.65 | R²: 0.7863


In [6]:
# ======================================================
# Baseline Model - Lasso Regression
# ======================================================
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.1)
lasso.fit(x_train, y_train)
y_val_pred_lasso = lasso.predict(x_val)
metrics_lasso = report_model("Lasso (baseline)", y_val, y_val_pred_lasso)

[Lasso (baseline)]  MAE: 2,908.59 | R²: 0.7863


/Users/karaca/src/MachineLearningProject-NOVAIMS2025/.venv/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.899e+09, tolerance: 5.648e+08
  model = cd_fast.enet_coordinate_descent(


In [7]:
# ======================================================
# Baseline Model - ElasticNet
# ======================================================

from sklearn.linear_model import ElasticNet


elasticnet = ElasticNet(alpha=0.1, l1_ratio=0.5)
elasticnet.fit(x_train, y_train)
y_val_pred_elasticnet = elasticnet.predict(x_val)
metrics_elasticnet = report_model("ElasticNet (baseline)", y_val, y_val_pred_elasticnet)


[ElasticNet (baseline)]  MAE: 2,985.07 | R²: 0.7753


In [8]:
# ======================================================
# Baseline Model — RandomForest
# ======================================================
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf.fit(x_train, y_train)

y_val_pred_rf = rf.predict(x_val)
metrics_rf = report_model("RandomForest (baseline)", y_val, y_val_pred_rf)


[RandomForest (baseline)]  MAE: 1,549.51 | R²: 0.9268


In [9]:
# ======================================================
# Baseline Model - DecisionTree
# ======================================================
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(
    max_depth=None,
    random_state=42,
)
dt.fit(x_train, y_train)
y_val_pred_dt = dt.predict(x_val)
metrics_dt = report_model("DecisionTree (baseline)", y_val, y_val_pred_dt)

[DecisionTree (baseline)]  MAE: 2,030.23 | R²: 0.8683


<a id="sec-4-comparison"></a>
## 4. Comparison of Models

In [10]:
# ======================================================
# Compare baselines & pick current best
# ======================================================
import pandas as pd

cmp = pd.DataFrame([
    {"model": "LinearRegression", **metrics_lin},
    {"model": "Ridge", **metrics_ridge},
    {"model": "Lasso", **metrics_lasso},
    {"model": "ElasticNet", **metrics_elasticnet},
    {"model": "RandomForest", **metrics_rf},
    {"model": "DecisionTree", **metrics_dt},
]).sort_values(by="MAE")

display(cmp)

best_name = cmp.iloc[0]["model"]
print(f"Current best (validation): {best_name}")


,model,MAE,R2
4,RandomForest,1549.508649,0.926758
5,DecisionTree,2030.229350,0.868307
2,Lasso,2908.594418,0.786348
1,Ridge,2908.650498,0.786344
0,LinearRegression,2908.688135,0.786341
3,ElasticNet,2985.072477,0.775267


Current best (validation): RandomForest


<a id="sec-5-hyperparameter-tuning"></a>
## 5. Hyperparameter Tuning & Selection

In [11]:
from sklearn.model_selection import ParameterGrid
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

param_grid = {
    "n_estimators": [200, 400, 600],
    "max_depth": [None, 20, 40],
    "max_features": ["sqrt", "log2", 0.5, 1],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

grid = list(ParameterGrid(param_grid))
n_total = len(grid)

best_score = float("inf")
best_params = None

for i, params in enumerate(grid, start=1):
    print(f"[{i}/{n_total}] Fitting with params: {params}")
    model = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
    model.fit(x_train, y_train)
    y_pred = model.predict(x_val)
    mae = mean_absolute_error(y_val, y_pred)

    print(f"    -> MAE: {mae:.3f}")

    if mae < best_score:
        best_score = mae
        best_params = params
        print(f"    ✅ New best params: {best_params} (MAE={best_score:.3f})")

print("\nBest params:", best_params)
print("Best MAE:", best_score)


[1/324] Fitting with params: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
    -> MAE: 1648.121
    ✅ New best params: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200} (MAE=1648.121)
[2/324] Fitting with params: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 400}
    -> MAE: 1643.487
    ✅ New best params: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 400} (MAE=1643.487)
[3/324] Fitting with params: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 600}
    -> MAE: 1642.490
    ✅ New best params: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 600} (MAE=1642.490)
[4/324] Fitting with params: {'max_depth': None, 'max_features':

<a id="sec-2-load"></a>
## 6. Submission File to Kaggle

In [12]:
# Build the best model 

# 2) Use the random forest model with the best hyperparameters found
best_model = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
best_model.fit(pd.concat([x_train, x_val], axis=0), pd.concat([y_train, y_val], axis=0))


,n_estimators,400
,criterion,'squared_error'
,max_depth,40
,min_samples_split,5
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,0.5
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [ ]:
# ======================================================
# Create Kaggle Submission (carID, price)
# ======================================================

# Load the raw test file to obtain carID
test_raw_path = os.path.join(data_dir, "test.csv")
test_raw = pd.read_csv(test_raw_path)

assert len(test_raw) == len(x_test), "Length mismatch between test.csv and x_test_final!"


y_test_pred = best_model.predict(x_test)

# (Optional) Round/clip according to competition rules
# Here we round to whole units, as in the sample, without allowing negative prices:
y_test_pred = np.clip(y_test_pred, a_min=0, a_max=None)
y_test_pred_rounded = np.rint(y_test_pred).astype(int)

# Build the submission DataFrame
submission = pd.DataFrame({
    "carID": test_raw["carID"],
    "price": y_test_pred_rounded  # optionally switch to y_test_pred if floats are allowed or preferred
})

# Save
sub_dir = os.path.join(data_dir, "submissions")
os.makedirs(sub_dir, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M")
sub_path = os.path.join(sub_dir, f"30_submission_{ts}.csv")
submission.to_csv(sub_path, index=False)

print(f"Submission saved to: {sub_path}")
display(submission.head(10))


int64
Submission saved to: ../data/submissions/30_submission_20251103_0302.csv


,carID,price
0,89856,13759
1,106581,24417
2,80886,13685
3,100174,16744
4,81376,26817
5,85391,10403
6,82175,14588
7,95250,16652
8,85071,5297
9,96210,17280


Pipeline to the Kaggle Competition Submission

- you need to set up your api key in the .env file as KAGGLE_USERNAME and KAGGLE_KEY

In [14]:
from dotenv import load_dotenv
from pathlib import Path
import os, json

env_path = Path("..") / ".env"   
load_dotenv(env_path, override=True)

kuser = os.getenv("KAGGLE_USERNAME")
kkey  = os.getenv("KAGGLE_KEY")

print("KAGGLE_USERNAME:", kuser)
print(".env loaded from:", env_path.resolve())


KAGGLE_USERNAME: mehmet1700
.env loaded from: /Users/karaca/src/MachineLearningProject-NOVAIMS2025/.env


In [15]:
!kaggle competitions submit -c cars4you -f {sub_path} -m "Group 21 submission"


100%|█████████████████████████████████████████| 383k/383k [00:01<00:00, 372kB/s]
Successfully submitted to Cars4you

In [16]:
# Get the results of the submission
!kaggle competitions submissions -c cars4you

fileName                         date                        description          status                     publicScore  privateScore  
-------------------------------  --------------------------  -------------------  -------------------------  -----------  ------------  
30_submission_20251103_0302.csv  2025-11-03 03:02:47.453000  Group 21 submission  SubmissionStatus.COMPLETE  1555.80496                 
30_submission_20251103_0228.csv  2025-11-03 02:28:21.043000  Group 21 submission  SubmissionStatus.COMPLETE  2218.30130                 
30_submission_20251103_0136.csv  2025-11-03 01:36:51.960000  Group 21 submission  SubmissionStatus.COMPLETE  2368.07167                 
30_submission_20251103_0135.csv  2025-11-03 01:35:44.233000  Group 21 submission  SubmissionStatus.COMPLETE  2368.07167                 
30_submission_20251103_0126.csv  2025-11-03 01:26:20.713000  Group 21 submission  SubmissionStatus.COMPLETE  2368.07167                 
30_submission_20251103_0111.csv  2025-11-

<a id="sec-2-load"></a>
## 7. Analysis of the predicted results

We had some predictions and results for our dataset. To improve further we want to find out, which entries are hard to predict and why.
For that we will try to analyze the absolute error for the validation dataset. For that we predict our random forest model on the validation set and calculate the absolute error. Then we merge it with the original validation data to have all features available for analysis. We look at the top entries with the highest absolute error.

In [34]:
"""
We had some predictions and results for our dataset. To improve further we want to find out, which entries are hard to predict and why.
For that we will try to analyze the absolute error for the validation dataset.

 For that we predict our random forest model on the validation set and calculate the absolute error. We will build it new on training dataset and calculate the absolute error.

 
 Then we merge it with the original validation data to have all features available for analysis. 
 We look at the top entries with the highest absolute error.



"""

# Build a new model on train data
final_model = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
final_model.fit(x_train, y_train)

# Predict on validation set
y_val_pred_final = final_model.predict(x_val)
# Calculate absolute error
abs_error = np.abs(y_val - y_val_pred_final)
# Merge with original validation data
val_analysis = x_val.copy()
val_analysis["actual_price"] = y_val
val_analysis["predicted_price"] = y_val_pred_final
val_analysis["absolute_error"] = abs_error

# Sort by absolute error descending
val_analysis_sorted = val_analysis.sort_values(by="absolute_error", ascending=False)
# Display top entries with highest absolute error
display(val_analysis_sorted.head(10))

,year,mileage,tax,mpg,engineSize,paintQuality%,previousOwners,Brand_Audi,Brand_BMW,Brand_Ford,...,transmission_Manual,transmission_Semi-Auto,fuelType_Diesel,fuelType_Hybrid,fuelType_Petrol,car_age,mileage_per_year,actual_price,predicted_price,absolute_error
6180,0.666667,-0.562096,0.00,-1.030769,-0.142857,0.628571,-0.5,0.0,0.0,0.0,...,-1.0,0.0,0.0,0.0,0.0,-0.666667,-0.651151,91874.0,23196.810282,68677.189718
1186,0.333333,-0.178930,0.00,-2.530769,3.428571,-0.800000,0.0,0.0,0.0,0.0,...,-1.0,1.0,0.0,0.0,0.0,-0.333333,-0.157520,139995.0,94388.166793,45606.833207
3598,0.666667,-0.676196,0.00,-1.861538,3.428571,0.485714,0.0,0.0,0.0,0.0,...,-1.0,1.0,0.0,0.0,0.0,-0.666667,-0.835075,115359.0,70502.371187,44856.628813
8668,0.666667,-0.623480,0.00,0.000000,-0.142857,0.085714,-1.0,0.0,1.0,0.0,...,-1.0,0.0,0.0,0.0,0.0,-0.666667,-0.750100,64750.0,21395.514029,43354.485971
5761,-0.666667,-0.304671,21.25,-2.600000,5.571429,0.285714,0.5,0.0,0.0,0.0,...,-1.0,1.0,0.0,0.0,0.0,0.666667,-0.502362,99850.0,59091.018645,40758.981355
1129,0.000000,-0.305001,-0.50,0.000000,-0.142857,0.142857,0.5,0.0,1.0,0.0,...,-1.0,0.0,0.0,1.0,-1.0,0.000000,-0.402948,59950.0,27905.819255,32044.180745
12790,0.666667,-0.565935,0.00,-1.738462,3.428571,0.885714,1.0,0.0,0.0,0.0,...,-1.0,0.0,0.0,0.0,0.0,-0.666667,-0.657340,104590.0,73679.807040,30910.192960
14834,0.666667,-0.623811,0.00,0.000000,4.000000,-0.114286,0.5,0.0,1.0,0.0,...,-1.0,1.0,0.0,0.0,0.0,-0.666667,-0.750632,89900.0,60426.061411,29473.938589
6843,0.333333,-0.509257,0.25,-2.407692,5.142857,0.114286,1.0,1.0,0.0,0.0,...,-1.0,0.0,0.0,0.0,0.0,-0.333333,-0.613925,95950.0,68436.111626,27513.888374
11161,-1.000000,0.984809,-7.25,0.000000,-0.142857,0.857143,-0.5,0.0,1.0,0.0,...,-1.0,0.0,0.0,1.0,-1.0,1.000000,0.595116,43497.0,17489.617317,26007.382683
